In this file, we
create multiplex horizontal visibility graphs for a certain time window in a file (edge list + adjacency matrix)

In [4]:
!pip install ts2vg

from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import save_npz
from ts2vg import HorizontalVG

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.2 MB/s eta 0:00:00


In [14]:
# path
BASE_DIR = Path("/content/epilepsy_pediatrics_EEG")

RAW_DIR = BASE_DIR / "data" / "raw"
GRAPH_DIR = BASE_DIR / "data" / "graphs"

GRAPH_DIR.mkdir(parents=True, exist_ok=True)

ictal_pkl_path = RAW_DIR / "/content/drive/MyDrive/Data/chb01_15_full_labeled.pkl"
interictal_pkl_path = RAW_DIR / "/content/drive/MyDrive/Data/chb01_01_preprocessed.pkl"

print("Ictal PKL exists      :", ictal_pkl_path.exists(), ictal_pkl_path)
print("Interictal PKL exists :", interictal_pkl_path.exists(), interictal_pkl_path)

Ictal PKL exists      : True /content/drive/MyDrive/Data/chb01_15_full_labeled.pkl
Interictal PKL exists : True /content/drive/MyDrive/Data/chb01_01_preprocessed.pkl


In [6]:
# LOAD PKL
def load_eeg_pickle(pkl_path):
    """
    Load a pickle file assumed to contain a pandas DataFrame.
    Ensures:
    - result is a DataFrame
    - time is in the index
    - index is numeric seconds if possible
    """
    obj = pd.read_pickle(pkl_path)

    if isinstance(obj, pd.DataFrame):
        df = obj.copy()
    else:
        raise TypeError(f"{pkl_path.name} does not contain a pandas DataFrame. Found: {type(obj)}")

    # If time is a column, move it to index
    possible_time_cols = ["Time (s)", "time", "time_s", "timestamp", "seconds"]
    found_time_col = None
    for c in possible_time_cols:
        if c in df.columns:
            found_time_col = c
            break

    if found_time_col is not None:
        df = df.set_index(found_time_col)

    # Try to ensure numeric index
    try:
        df.index = pd.to_numeric(df.index)
    except Exception:
        pass

    # sort by time just in case
    try:
        df = df.sort_index()
    except Exception:
        pass

    return df

In [7]:
def inspect_df(df, name="dataframe"):
    print(f"\n===== {name} =====")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print("Index type:", type(df.index))
    print("Index start:", df.index[0])
    print("Index end  :", df.index[-1])

    if "label" in df.columns:
        print("Label counts:")
        print(df["label"].value_counts(dropna=False))
    else:
        print("No 'label' column found.")

In [8]:
# EXTRACT FIXED WINDOW
def extract_window_by_time(df, start_time_s, window_size_s=5, expected_label=None):
    """
    Extract a fixed window by absolute time in seconds.

    Parameters
    ----------
    df : pd.DataFrame
        EEG dataframe indexed by time in seconds.
    start_time_s : float
        Desired start time.
    window_size_s : int or float
        Window size in seconds.
    expected_label : str or None
        If provided, checks that all rows have this label.

    Returns
    -------
    window_df : pd.DataFrame
    start_idx : int
    """
    if len(df) < 2:
        raise ValueError("DataFrame is too short to infer sampling rate.")

    dt = float(df.index[1] - df.index[0])
    sfreq = int(round(1 / dt))
    window_size_samples = int(round(window_size_s * sfreq))

    # closest time index to requested start
    index_values = np.asarray(df.index, dtype=float)
    start_idx = int(np.argmin(np.abs(index_values - start_time_s)))
    end_idx = start_idx + window_size_samples

    if end_idx > len(df):
        raise ValueError(
            f"Requested window [{start_time_s}, {start_time_s + window_size_s}] exceeds file length."
        )

    window_df = df.iloc[start_idx:end_idx].copy()

    if expected_label is not None:
        if "label" not in window_df.columns:
            raise ValueError(f"expected_label='{expected_label}' was requested, but no 'label' column exists.")
        if not (window_df["label"] == expected_label).all():
            found = window_df["label"].value_counts(dropna=False).to_dict()
            raise ValueError(
                f"Window {start_time_s:.3f}-{start_time_s + window_size_s:.3f}s is not fully '{expected_label}'. "
                f"Found labels: {found}"
            )

    return window_df, start_idx

In [9]:
# node table
def make_multiplex_node_table(window_df, graph_label):
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_timepoints = len(window_df)

    rows = []
    for electrode in electrode_cols:
        for t in range(n_timepoints):
            node_name = f"{electrode}_t{t}"
            rows.append({
                "Id": node_name,
                "Label": node_name,
                "electrode": electrode,
                "time_idx": t,
                "layer": electrode,
                "graph_label": graph_label
            })

    return pd.DataFrame(rows)

In [10]:
# BUILD MULTIPLEX HVG
def build_multiplex_hvg(window_df):
    """
    Build multiplex HVG:
    - intra-layer: HVG within each electrode
    - inter-layer: full clique across electrodes at each time point
    """
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_electrodes = len(electrode_cols)
    n_timepoints = len(window_df)
    n_nodes_total = n_electrodes * n_timepoints

    print(f"Electrodes        : {n_electrodes}")
    print(f"Time points       : {n_timepoints}")
    print(f"Total nodes       : {n_nodes_total}")

    def node_id(electrode, t):
        return f"{electrode}_t{t}"

    node_labels = [
        node_id(electrode, t)
        for electrode in electrode_cols
        for t in range(n_timepoints)
    ]

    label_to_idx = {label: idx for idx, label in enumerate(node_labels)}

    node_index = {
        (electrode, t): node_id(electrode, t)
        for electrode in electrode_cols
        for t in range(n_timepoints)
    }

    adj_sparse = sp.lil_matrix((n_nodes_total, n_nodes_total), dtype=np.int8)

    # 1) intra-layer edges
    intra_rows = []
    layer_info = {}

    for layer_idx, electrode in enumerate(electrode_cols):
        ts = window_df[electrode].values
        hvg = HorizontalVG()
        hvg.build(ts)

        for (t_i, t_j) in hvg.edges:
            u = node_id(electrode, t_i)
            v = node_id(electrode, t_j)

            intra_rows.append({
                "Source": u,
                "Target": v,
                "source_label": u,
                "target_label": v,
                "edge_type": "intra",
                "layer": electrode
            })

            i = label_to_idx[u]
            j = label_to_idx[v]
            adj_sparse[i, j] = 1
            adj_sparse[j, i] = 1

        layer_info[electrode] = {
            "n_intra_edges": len(hvg.edges),
            "layer_index": layer_idx
        }

    print(f"Intra-layer edges : {len(intra_rows)}")

    # 2) inter-layer edges
    inter_rows = []

    for t in range(n_timepoints):
        for i in range(n_electrodes):
            for j in range(i + 1, n_electrodes):
                e_i = electrode_cols[i]
                e_j = electrode_cols[j]

                u = node_id(e_i, t)
                v = node_id(e_j, t)

                inter_rows.append({
                    "Source": u,
                    "Target": v,
                    "source_label": u,
                    "target_label": v,
                    "edge_type": "inter",
                    "layer": f"{e_i}<->{e_j}"
                })

                ui = label_to_idx[u]
                vj = label_to_idx[v]
                adj_sparse[ui, vj] = 1
                adj_sparse[vj, ui] = 1

    print(f"Inter-layer edges : {len(inter_rows)}")

    edge_list = pd.DataFrame(intra_rows + inter_rows)
    print(f"Total edges       : {len(edge_list)}")

    return edge_list, adj_sparse.tocsr(), node_index, layer_info, node_labels

In [11]:
def save_multiplex_hvg_outputs(
    edge_list,
    adj_sparse,
    node_index,
    node_labels,
    layer_info,
    window_df,
    graph_id,
    source_file,
    window_id,
    label,
    out_root="data/graphs"
):
    out_root = Path(out_root)
    edges_dir = out_root / "edges"
    nodes_dir = out_root / "nodes"
    adj_dir = out_root / "adjacency_sparse"
    meta_dir = out_root / "metadata"
    windows_dir = out_root / "windows"

    for d in [edges_dir, nodes_dir, adj_dir, meta_dir, windows_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # node table
    node_df = make_multiplex_node_table(window_df, graph_label=label)

    # save edge list
    edge_path = edges_dir / f"{graph_id}_edges.csv"
    edge_list.to_csv(edge_path, index=False)

    # save node table
    node_path = nodes_dir / f"{graph_id}_nodes.csv"
    node_df.to_csv(node_path, index=False)

    # save adjacency
    adj_path = adj_dir / f"{graph_id}_adjacency_sparse.npz"
    save_npz(adj_path, adj_sparse)

    # save window signal itself
    window_path = windows_dir / f"{graph_id}_window.parquet"
    window_df.to_parquet(window_path)

    # save node labels
    node_labels_path = meta_dir / f"{graph_id}_node_labels.json"
    with open(node_labels_path, "w", encoding="utf-8") as f:
        json.dump(node_labels, f, indent=2)

    # save layer info
    layer_info_df = (
        pd.DataFrame(layer_info)
        .T
        .reset_index()
        .rename(columns={"index": "electrode"})
    )
    layer_info_path = meta_dir / f"{graph_id}_layer_info.csv"
    layer_info_df.to_csv(layer_info_path, index=False)

    # save node index
    node_index_str = {f"{k[0]}__t{k[1]}": v for k, v in node_index.items()}
    node_index_path = meta_dir / f"{graph_id}_node_index.json"
    with open(node_index_path, "w", encoding="utf-8") as f:
        json.dump(node_index_str, f, indent=2)

    # save metadata
    metadata = {
        "graph_id": graph_id,
        "source_file": str(source_file),
        "window_id": window_id,
        "label": label,
        "n_nodes": int(node_df.shape[0]),
        "n_edges": int(edge_list.shape[0]),
        "n_intra_edges": int((edge_list["edge_type"] == "intra").sum()),
        "n_inter_edges": int((edge_list["edge_type"] == "inter").sum()),
        "n_layers": int(node_df["electrode"].nunique()),
        "n_timepoints": int(len(window_df)),
        "window_start_time_s": float(window_df.index[0]),
        "window_end_time_s": float(window_df.index[-1]),
        "adjacency_format": "scipy_sparse_npz"
    }

    metadata_path = meta_dir / f"{graph_id}_metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    return {
        "edges": str(edge_path),
        "nodes": str(node_path),
        "adjacency_sparse": str(adj_path),
        "window": str(window_path),
        "node_labels": str(node_labels_path),
        "layer_info": str(layer_info_path),
        "node_index": str(node_index_path),
        "metadata": str(metadata_path),
    }

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
# load pkl files
df_ictal = load_eeg_pickle(ictal_pkl_path)
df_interictal = load_eeg_pickle(interictal_pkl_path)

inspect_df(df_ictal, "chb01_15_full_labeled.pkl") # connect to your own path (e.g. github data folder)
inspect_df(df_interictal, "chb01_01_preprocessed.pkl") # connect to your own path


===== chb01_15_full_labeled.pkl =====
Shape: (921600, 24)
Columns: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1', 'label']
Index type: <class 'pandas.core.indexes.base.Index'>
Index start: 0.0
Index end  : 3599.99609375
Label counts:
label
interictal    911359
ictal          10241
Name: count, dtype: int64

===== chb01_01_preprocessed.pkl =====
Shape: (921600, 23)
Columns: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Index type: <class 'pandas.core.indexes.base.Index'>
Index start: 0.0
Index end  : 3599.99609375
No 'label' column found.


In [16]:
# ---------------------------------------------------------
# WINDOW SPECS
# ---------------------------------------------------------
window_size_s = 5

window_specs = [
    {
        "source_name": "chb01_15_full_labeled",
        "df": df_ictal,
        "source_file": ictal_pkl_path,
        "window_id": "ictal_1735_1740",
        "start_time_s": 1735.0,
        "label": "ictal",
        "expected_label": "ictal"
    },
    {
        "source_name": "chb01_15_full_labeled",
        "df": df_ictal,
        "source_file": ictal_pkl_path,
        "window_id": "ictal_1740_1745",
        "start_time_s": 1740.0,
        "label": "ictal",
        "expected_label": "ictal"
    },
    {
        "source_name": "chb01_15_full_labeled",
        "df": df_ictal,
        "source_file": ictal_pkl_path,
        "window_id": "ictal_1745_1750",
        "start_time_s": 1745.0,
        "label": "ictal",
        "expected_label": "ictal"
    },
    {
        "source_name": "chb01_01_preprocessed",
        "df": df_interictal,
        "source_file": interictal_pkl_path,
        "window_id": "interictal_3110_3115",
        "start_time_s": 3110.0,
        "label": "interictal",
        "expected_label": "interictal" if "label" in df_interictal.columns else None
    },
    {
        "source_name": "chb01_01_preprocessed",
        "df": df_interictal,
        "source_file": interictal_pkl_path,
        "window_id": "interictal_3115_3120",
        "start_time_s": 3115.0,
        "label": "interictal",
        "expected_label": "interictal" if "label" in df_interictal.columns else None
    },
    {
        "source_name": "chb01_01_preprocessed",
        "df": df_interictal,
        "source_file": interictal_pkl_path,
        "window_id": "interictal_3120_3125",
        "start_time_s": 3120.0,
        "label": "interictal",
        "expected_label": "interictal" if "label" in df_interictal.columns else None
    },
]

In [17]:
# ---------------------------------------------------------
# EXTRACT ALL WINDOWS
# ---------------------------------------------------------
windows = {}
windows_index_records = []

for spec in window_specs:
    window_df, start_idx = extract_window_by_time(
        df=spec["df"],
        start_time_s=spec["start_time_s"],
        window_size_s=window_size_s,
        expected_label=spec["expected_label"]
    )

    windows[spec["window_id"]] = {
        "window_df": window_df,
        "source_file": spec["source_file"],
        "label": spec["label"],
        "source_name": spec["source_name"]
    }

    windows_index_records.append({
        "window_id": spec["window_id"],
        "source_name": spec["source_name"],
        "source_file": str(spec["source_file"]),
        "label": spec["label"],
        "requested_start_time_s": spec["start_time_s"],
        "actual_start_time_s": float(window_df.index[0]),
        "actual_end_time_s": float(window_df.index[-1]),
        "start_idx": int(start_idx),
        "end_idx": int(start_idx + len(window_df) - 1),
        "n_samples": int(len(window_df)),
        "window_size_s": window_size_s
    })

    print(f"[OK] {spec['window_id']}")
    print(f"     requested start : {spec['start_time_s']}")
    print(f"     actual start    : {float(window_df.index[0])}")
    print(f"     actual end      : {float(window_df.index[-1])}")
    print(f"     source          : {spec['source_name']}")
    print(f"     label           : {spec['label']}")

windows_index = pd.DataFrame(windows_index_records)
display(windows_index)

[OK] ictal_1735_1740
     requested start : 1735.0
     actual start    : 1735.0
     actual end      : 1739.99609375
     source          : chb01_15_full_labeled
     label           : ictal
[OK] ictal_1740_1745
     requested start : 1740.0
     actual start    : 1740.0
     actual end      : 1744.99609375
     source          : chb01_15_full_labeled
     label           : ictal
[OK] ictal_1745_1750
     requested start : 1745.0
     actual start    : 1745.0
     actual end      : 1749.99609375
     source          : chb01_15_full_labeled
     label           : ictal
[OK] interictal_3110_3115
     requested start : 3110.0
     actual start    : 3110.0
     actual end      : 3114.99609375
     source          : chb01_01_preprocessed
     label           : interictal
[OK] interictal_3115_3120
     requested start : 3115.0
     actual start    : 3115.0
     actual end      : 3119.99609375
     source          : chb01_01_preprocessed
     label           : interictal
[OK] interictal_3120

,window_id,source_name,source_file,label,requested_start_time_s,actual_start_time_s,actual_end_time_s,start_idx,end_idx,n_samples,window_size_s
0,ictal_1735_1740,chb01_15_full_labeled,/content/drive/MyDrive/Data/chb01_15_full_labe...,ictal,1735.0,1735.0,1739.996094,444160,445439,1280,5
1,ictal_1740_1745,chb01_15_full_labeled,/content/drive/MyDrive/Data/chb01_15_full_labe...,ictal,1740.0,1740.0,1744.996094,445440,446719,1280,5
2,ictal_1745_1750,chb01_15_full_labeled,/content/drive/MyDrive/Data/chb01_15_full_labe...,ictal,1745.0,1745.0,1749.996094,446720,447999,1280,5
3,interictal_3110_3115,chb01_01_preprocessed,/content/drive/MyDrive/Data/chb01_01_preproces...,interictal,3110.0,3110.0,3114.996094,796160,797439,1280,5
4,interictal_3115_3120,chb01_01_preprocessed,/content/drive/MyDrive/Data/chb01_01_preproces...,interictal,3115.0,3115.0,3119.996094,797440,798719,1280,5
5,interictal_3120_3125,chb01_01_preprocessed,/content/drive/MyDrive/Data/chb01_01_preproces...,interictal,3120.0,3120.0,3124.996094,798720,799999,1280,5


In [18]:
# ---------------------------------------------------------
# BUILD + SAVE ALL 6 GRAPHS
# ---------------------------------------------------------
all_saved_paths = {}

for window_id, item in windows.items():
    window_df = item["window_df"]
    graph_label = item["label"]
    source_file = item["source_file"]

    graph_id = window_id

    print("\n" + "=" * 70)
    print(f"Building graph: {graph_id}")
    print("=" * 70)

    edge_list, adj_sparse, node_index, layer_info, node_labels = build_multiplex_hvg(window_df)

    saved_paths = save_multiplex_hvg_outputs(
        edge_list=edge_list,
        adj_sparse=adj_sparse,
        node_index=node_index,
        node_labels=node_labels,
        layer_info=layer_info,
        window_df=window_df,
        graph_id=graph_id,
        source_file=source_file,
        window_id=window_id,
        label=graph_label,
        out_root=GRAPH_DIR
    )

    all_saved_paths[window_id] = saved_paths

print("\nAll 6 graphs saved successfully.")
for k, v in all_saved_paths.items():
    print(f"\n{k}")
    print(v["edges"])


Building graph: ictal_1735_1740
Electrodes        : 23
Time points       : 1280
Total nodes       : 29440
Intra-layer edges : 56965
Inter-layer edges : 323840
Total edges       : 380805

Building graph: ictal_1740_1745
Electrodes        : 23
Time points       : 1280
Total nodes       : 29440
Intra-layer edges : 57393
Inter-layer edges : 323840
Total edges       : 381233

Building graph: ictal_1745_1750
Electrodes        : 23
Time points       : 1280
Total nodes       : 29440
Intra-layer edges : 57315
Inter-layer edges : 323840
Total edges       : 381155

Building graph: interictal_3110_3115
Electrodes        : 23
Time points       : 1280
Total nodes       : 29440
Intra-layer edges : 58292
Inter-layer edges : 323840
Total edges       : 382132

Building graph: interictal_3115_3120
Electrodes        : 23
Time points       : 1280
Total nodes       : 29440
Intra-layer edges : 58293
Inter-layer edges : 323840
Total edges       : 382133

Building graph: interictal_3120_3125
Electrodes       

In [28]:
from pathlib import Path

subdirs = ["edges", "nodes", "adjacency_sparse", "windows", "metadata"]

for sub in subdirs:
    folder = Path(GRAPH_DIR) / sub
    files = sorted(folder.glob("*"))

    print(f"\n📁 {sub} ({len(files)} files)")
    print("-" * 50)

    for f in files:
        print(f.name)


📁 edges (6 files)
--------------------------------------------------
ictal_1735_1740_edges.csv
ictal_1740_1745_edges.csv
ictal_1745_1750_edges.csv
interictal_3110_3115_edges.csv
interictal_3115_3120_edges.csv
interictal_3120_3125_edges.csv

📁 nodes (6 files)
--------------------------------------------------
ictal_1735_1740_nodes.csv
ictal_1740_1745_nodes.csv
ictal_1745_1750_nodes.csv
interictal_3110_3115_nodes.csv
interictal_3115_3120_nodes.csv
interictal_3120_3125_nodes.csv

📁 adjacency_sparse (6 files)
--------------------------------------------------
ictal_1735_1740_adjacency_sparse.npz
ictal_1740_1745_adjacency_sparse.npz
ictal_1745_1750_adjacency_sparse.npz
interictal_3110_3115_adjacency_sparse.npz
interictal_3115_3120_adjacency_sparse.npz
interictal_3120_3125_adjacency_sparse.npz

📁 windows (6 files)
--------------------------------------------------
ictal_1735_1740_window.parquet
ictal_1740_1745_window.parquet
ictal_1745_1750_window.parquet
interictal_3110_3115_window.parquet